In [50]:
import pandas as pd
import ast
import re
from sklearn.feature_extraction.text import CountVectorizer
from scipy.cluster.hierarchy import linkage, fcluster
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import AgglomerativeClustering
import faiss
import numpy as np



In [51]:

df = pd.read_csv('C:/Users/k.hegde/Desktop/Learning_Projects/Recipe_Recommender/data/1_Recipe_csv.csv')
print(df.head())

                                        recipe_title           category  \
0         Air Fryer Potato Slices with Dipping Sauce  Air Fryer Recipes   
1                         Gochujang Pork Belly Bites  Air Fryer Recipes   
2  3-Ingredient Air Fryer Everything Bagel Chicke...  Air Fryer Recipes   
3         Air Fryer Everything Bagel Chicken Cutlets  Air Fryer Recipes   
4              Air Fryer Honey Sriracha Salmon Bites  Air Fryer Recipes   

         subcategory                                        description  \
0  Air Fryer Recipes  These air fryer potato slices, served with a b...   
1  Air Fryer Recipes  These gochujang pork belly bites are sweet and...   
2  Air Fryer Recipes  These 3-ingredient air fryer everything bagel ...   
3  Air Fryer Recipes  These air fryer everything bagel chicken cutle...   
4  Air Fryer Recipes  These air fryer honey Sriracha salmon bites ar...   

                                         ingredients  \
0  ["3/4 cup ketchup", "1/2 cup beer", "1 

In [52]:
df['ingredients'] = df['ingredients'].apply(ast.literal_eval)
df['directions'] = df['directions'].apply(ast.literal_eval)

df.head()

,recipe_title,category,subcategory,description,ingredients,directions,num_ingredients,num_steps
0,Air Fryer Potato Slices with Dipping Sauce,Air Fryer Recipes,Air Fryer Recipes,"These air fryer potato slices, served with a b...","[3/4 cup ketchup, 1/2 cup beer, 1 tablespoon W...","[Combine ketchup, beer, Worcestershire sauce, ...",9,5
1,Gochujang Pork Belly Bites,Air Fryer Recipes,Air Fryer Recipes,These gochujang pork belly bites are sweet and...,"[1 pound pork belly, 1/4 cup gochujang, 2 tabl...",[Preheat an air fryer to 400 degrees F (200 de...,5,4
2,3-Ingredient Air Fryer Everything Bagel Chicke...,Air Fryer Recipes,Air Fryer Recipes,These 3-ingredient air fryer everything bagel ...,"[1 ¼ pounds chicken tenders, 1 tablespoon oliv...",[Gather all ingredients. Preheat an air fryer ...,3,4
3,Air Fryer Everything Bagel Chicken Cutlets,Air Fryer Recipes,Air Fryer Recipes,These air fryer everything bagel chicken cutle...,"[4 chicken cutlets (about 1 pound total), salt...",[Preheat an air fryer to 400 degrees F (200 de...,9,9
4,Air Fryer Honey Sriracha Salmon Bites,Air Fryer Recipes,Air Fryer Recipes,These air fryer honey Sriracha salmon bites ar...,"[1 tablespoon soy sauce, 1 tablespoon honey, 1...",[Preheat an air fryer to 400 degrees F (200 de...,5,5


In [ ]:

units = ['cup', 'cups', 'pound', 'pounds', 'tablespoon', 'tablespoons',
         'teaspoon', 'teaspoons', 'oz', 'ounce', 'ounces', 'gram', 'grams',
         'kg', 'ml', 'liter', 'liters', 'pinch', 'dash']

def clean_ingredient_list(ingredient_list):
    cleaned = []
    for ing in ingredient_list:
        ing = ing.lower()
        ing = re.sub(r'[^a-zA-Z ]', '', ing)
        words = ing.split()
        filtered = [word for word in words if word not in units]
        cleaned.append(' '.join(filtered))
    return ' '.join(cleaned)  # Join all ingredients into one string

df['clean_ingredients'] = df['ingredients'].apply(clean_ingredient_list)

df.head()

,recipe_title,category,subcategory,description,ingredients,directions,num_ingredients,num_steps,clean_ingredients
0,Air Fryer Potato Slices with Dipping Sauce,Air Fryer Recipes,Air Fryer Recipes,"These air fryer potato slices, served with a b...","[3/4 cup ketchup, 1/2 cup beer, 1 tablespoon W...","[Combine ketchup, beer, Worcestershire sauce, ...",9,5,ketchup beer worcestershire sauce onion powder...
1,Gochujang Pork Belly Bites,Air Fryer Recipes,Air Fryer Recipes,These gochujang pork belly bites are sweet and...,"[1 pound pork belly, 1/4 cup gochujang, 2 tabl...",[Preheat an air fryer to 400 degrees F (200 de...,5,4,pork belly gochujang soy sauce honey ground gi...
2,3-Ingredient Air Fryer Everything Bagel Chicke...,Air Fryer Recipes,Air Fryer Recipes,These 3-ingredient air fryer everything bagel ...,"[1 ¼ pounds chicken tenders, 1 tablespoon oliv...",[Gather all ingredients. Preheat an air fryer ...,3,4,chicken tenders olive oil everything bagel sea...
3,Air Fryer Everything Bagel Chicken Cutlets,Air Fryer Recipes,Air Fryer Recipes,These air fryer everything bagel chicken cutle...,"[4 chicken cutlets (about 1 pound total), salt...",[Preheat an air fryer to 400 degrees F (200 de...,9,9,chicken cutlets about total salt and freshly g...
4,Air Fryer Honey Sriracha Salmon Bites,Air Fryer Recipes,Air Fryer Recipes,These air fryer honey Sriracha salmon bites ar...,"[1 tablespoon soy sauce, 1 tablespoon honey, 1...",[Preheat an air fryer to 400 degrees F (200 de...,5,5,soy sauce honey sriracha rice vinegar granulat...


In [54]:
def parse_directions(dir_data):
    # Handle NaN / None
    if dir_data is None:
        return []
    if isinstance(dir_data, float) and np.isnan(dir_data):
        return []
    
    # If numpy array → convert to list
    if isinstance(dir_data, np.ndarray):
        dir_data = dir_data.tolist()
    
    # If already a list
    if isinstance(dir_data, list):
        return [str(step).strip() for step in dir_data if str(step).strip()]
    
    # If string
    if isinstance(dir_data, str):
        dir_data = dir_data.strip()
        if dir_data == '':
            return []
        # Try to parse as list
        if dir_data.startswith('[') and dir_data.endswith(']'):
            try:
                parsed = ast.literal_eval(dir_data)
                if isinstance(parsed, list):
                    return [str(step).strip() for step in parsed if str(step).strip()]
            except:
                pass
        # Otherwise split by comma
        return [step.strip() for step in dir_data.split(',') if step.strip()]
    
    return []

# Apply safely
df['directions_clean'] = df['directions'].apply(parse_directions)


In [55]:
#hybrid approach
# Combine ingredients + directions into one text field
df['combined_text'] = df['clean_ingredients'] + ' ' + df['directions_clean'].apply(lambda steps: ' '.join(steps))


In [56]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Create a TF-IDF vectorizer for combined text
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = vectorizer.fit_transform(df['combined_text'])

In [57]:
# Reduce dimensions for FAISS
svd = TruncatedSVD(n_components=300, random_state=42)
X_reduced = svd.fit_transform(X.astype(np.float32))

# Normalize for cosine similarity
faiss.normalize_L2(X_reduced)


In [58]:
dim = X_reduced.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(X_reduced)
print("✅ FAISS index built for hybrid ingredients+directions.")

✅ FAISS index built for hybrid ingredients+directions.


In [59]:
def recommend_hybrid_recipes(input_ingredients, input_directions=None, top_k=5):
    """
    Recommend recipes based on user ingredients + optional directions,
    showing similarity, missing ingredients, and match count.
    """
    # 1️⃣ Prepare input text
    input_clean = clean_ingredient_list(input_ingredients)
    input_text = input_clean
    if input_directions:
        input_text += ' ' + ' '.join(input_directions)
    
    input_set = set(input_clean.split())

    # 2️⃣ Vectorize + reduce + normalize
    input_vector = vectorizer.transform([input_text]).astype(np.float32)
    input_reduced = svd.transform(input_vector)
    faiss.normalize_L2(input_reduced)

    # 3️⃣ Search FAISS
    distances, indices = index.search(input_reduced, top_k)

    # 4️⃣ Prepare recommendations
    recommendations = []
    for idx, sim in zip(indices[0], distances[0]):
        recipe_title = df.iloc[idx]['recipe_title']
        recipe_ing = df.iloc[idx]['clean_ingredients']
        recipe_set = set(recipe_ing.split())

        missing = recipe_set - input_set
        matched_count = len(recipe_set & input_set)
        total_count = len(recipe_set)

        recommendations.append({
            "recipe_title": recipe_title,
            "similarity": round(float(sim), 3),
            "matched_ingredients": matched_count,
            "total_ingredients": total_count,
            "missing_ingredients": list(missing),
            "directions": df.iloc[idx]['directions_clean']
        })

    return pd.DataFrame(recommendations).sort_values(by='similarity', ascending=False)


In [60]:
input_ingredients = ["chicken", "soy sauce", "honey"]


results = recommend_hybrid_recipes(input_ingredients, top_k=5)

results_clean=results.drop_duplicates(subset=['recipe_title'])
print(results_clean)


                       recipe_title  similarity  matched_ingredients  \
0      Slow Cooker Hawaiian Chicken       0.644                    4   
1     Orange, Honey and Soy Chicken       0.624                    4   
3  Sweet, Sticky, and Spicy Chicken       0.620                    4   

   total_ingredients                                missing_ingredients  \
0                 10  [can, barbecue, bottle, sliced, breast, pineap...   
1                 19  [boneless, oranges, black, pepper, juiced, gro...   
3                 27  [vegetable, boneless, pepper, halves, root, in...   

                                          directions  
0  [Put chicken breasts into the crock of a slow ...  
1  [Combine chicken, orange juice, soy sauce, hon...  
3  [Gather all ingredients., Whisk soy sauce, hon...  


In [61]:
import joblib
import os
os.makedirs("../models", exist_ok=True)

joblib.dump(vectorizer, "../models/hybrid_vectorizer.pkl")
joblib.dump(X, "../models/hybrid_fitted_vectorizer.pkl")
joblib.dump(X_reduced, "../models/hybrid_fitted_svd.pkl")
joblib.dump(svd, "../models/hybrid_svd.pkl")
faiss.write_index(index, "../models/hybrid_faiss_index.faiss")
df[['recipe_title', 'clean_ingredients', 'directions_clean']].to_csv("../models/recipes_meta.csv", index=False)


print("✅ Hybrid model and data saved successfully!")

✅ Hybrid model and data saved successfully!
